In [3]:
# Verifikasi RDKit
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')
print("RDKit OK ✅")

RDKit OK ✅


In [ ]:
# ======================
# STEP 0 — KONFIGURASI & EKSTRAKSI MORGAN (2048 bit, int 0/1)
# ======================
import pandas as pd
import numpy as np
import csv
from scipy import sparse
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

# --- Sumber data (Google Sheets → CSV) ---
SHEET_path = "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/data/SIDER/sider.csv"

# Parse kolom secara manual karena header di CSV kemungkinan terbungkus satu quote utuh
cols_text = 'smiles , hepatobiliary disorders, metabolism and nutrition disorders, product issues, eye disorders, investigations, musculoskeletal and connective tissue disorders, gastrointestinal disorders, social circumstances, immune system disorders, reproductive system and breast disorders, "neoplasms benign, malignant and unspecified (incl cysts and polyps)", general disorders and administration site conditions, endocrine disorders, surgical and medical procedures, vascular disorders, blood and lymphatic system disorders, skin and subcutaneous tissue disorders, "congenital, familial and genetic disorders", infections and infestations, "respiratory, thoracic and mediastinal disorders", psychiatric disorders, renal and urinary disorders, "pregnancy, puerperium and perinatal conditions", ear and labyrinth disorders, cardiac disorders, nervous system disorders, "injury, poisoning and procedural complications"'
parsed_cols = next(csv.reader([cols_text], skipinitialspace=True))
# Bersihkan spasi di nama kolom
parsed_cols = [c.strip().lower() for c in parsed_cols]

# --- Load dataset dengan list kolom eksplisit, skip row pertama (header yang error) ---
df = pd.read_csv(
    SHEET_path,
    sep=",",
    engine="python",
    header=0,
    names=parsed_cols
)

assert 'smiles' in df.columns, "Kolom 'smiles' wajib ada."

# --- Morgan → integer bits (0/1) ---
MORGAN_BITS   = 2048
MORGAN_RADIUS = 2

def morgan_bits_int(smiles: str, nBits: int = MORGAN_BITS, radius: int = MORGAN_RADIUS) -> np.ndarray:
    if pd.isna(smiles):
        return np.zeros(nBits, dtype=np.uint8)
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return np.zeros(nBits, dtype=np.uint8)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# Ekstraksi semua SMILES → CSR matrix uint8
X_arr = np.vstack([morgan_bits_int(s) for s in df["smiles"]])
X_INT = sparse.csr_matrix(X_arr, dtype=np.uint8)

# --- Groups (anti-leak) ---
GROUPS = df["group_id"].values if "group_id" in df.columns else np.arange(len(df))

# --- Deteksi label biner & pilih 3 label representatif (IR: besar/median/kecil) ---
NON_LABELS = ["smiles", "char", "group_id"]

def is_binary_series(s: pd.Series) -> bool:
    vals = pd.Series(s).dropna().astype(int).unique().tolist()
    return set(vals).issubset({0, 1}) and len(vals) >= 1

label_cols = [c for c in df.columns if c not in NON_LABELS and is_binary_series(df[c])]
assert len(label_cols) >= 1, "Tidak ada kolom label biner 0/1 yang terdeteksi."

def imbalance_ratio(colname: str) -> float:
    pos = int(df[colname].sum()); neg = int(len(df) - pos)
    if min(pos, neg) == 0:
        return float("inf")
    return max(pos, neg) / min(pos, neg)

labels_sorted = sorted(label_cols, key=imbalance_ratio, reverse=True)
if len(labels_sorted) >= 3:
    largest_ir_label  = labels_sorted[0]
    median_ir_label   = labels_sorted[len(labels_sorted)//2]
    smallest_ir_label = labels_sorted[-1]
    SELECTED_LABELS = [largest_ir_label, median_ir_label, smallest_ir_label]
else:
    SELECTED_LABELS = labels_sorted

def _ir_str(lbl):
    val = imbalance_ratio(lbl)
    return "inf" if np.isinf(val) else f"{val:.3f}"

print("X_INT shape:", X_INT.shape, "| dtype:", X_INT.dtype)
print("Total sampel:", len(df))
print("SELECTED_LABELS (IR besar/median/kecil):")
for lbl in SELECTED_LABELS:
    print(f"  - {lbl} (IR={_ir_str(lbl)})")

X_INT shape: (1427, 2048) | dtype: uint8
Total sampel: 1427
SELECTED_LABELS (IR besar/median/kecil):
  - product issues (IR=63.864)
  - neoplasms benign, malignant and unspecified (incl cysts and polyps) (IR=2.795)
  - reproductive system and breast disorders (IR=1.039)


In [5]:
# ====================================
# STEP 1 — BACA DATASET + VALIDASI
# ====================================
import pandas as pd
import csv

# Gunakan cara parsing yang sama seperti di STEP 0 untuk menghindari masalah header CSV
cols_text = 'smiles , hepatobiliary disorders, metabolism and nutrition disorders, product issues, eye disorders, investigations, musculoskeletal and connective tissue disorders, gastrointestinal disorders, social circumstances, immune system disorders, reproductive system and breast disorders, "neoplasms benign, malignant and unspecified (incl cysts and polyps)", general disorders and administration site conditions, endocrine disorders, surgical and medical procedures, vascular disorders, blood and lymphatic system disorders, skin and subcutaneous tissue disorders, "congenital, familial and genetic disorders", infections and infestations, "respiratory, thoracic and mediastinal disorders", psychiatric disorders, renal and urinary disorders, "pregnancy, puerperium and perinatal conditions", ear and labyrinth disorders, cardiac disorders, nervous system disorders, "injury, poisoning and procedural complications"'
parsed_cols = [c.strip().lower() for c in next(csv.reader([cols_text], skipinitialspace=True))]

df = pd.read_csv(
    SHEET_path,
    sep=",",
    engine="python",
    header=0,
    names=parsed_cols
)

print("=== INFO DATA ===")
df.info()
print("\n=== SHAPE ===", df.shape)
print("\n=== KOLOM (awal) ===", df.columns.tolist()[:25])

# Wajib ada 'smiles'
if "smiles" not in df.columns:
    raise ValueError("Kolom 'smiles' tidak ditemukan di spreadsheet.")

# Drop baris tanpa SMILES valid
before = len(df)
df = df[df["smiles"].astype(str).str.len() > 0].copy()
print(f"[INFO] Hapus {before - len(df)} baris SMILES kosong/invalid.")

=== INFO DATA ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1427 entries, 0 to 1426
Data columns (total 28 columns):
 #   Column                                                               Non-Null Count  Dtype 
---  ------                                                               --------------  ----- 
 0   smiles                                                               1427 non-null   object
 1   hepatobiliary disorders                                              1427 non-null   int64 
 2   metabolism and nutrition disorders                                   1427 non-null   int64 
 3   product issues                                                       1427 non-null   int64 
 4   eye disorders                                                        1427 non-null   int64 
 5   investigations                                                       1427 non-null   int64 
 6   musculoskeletal and connective tissue disorders                      1427 non-null   int64 
 7

In [6]:
# ==================================================
# STEP 2 — DETEKSI LABEL BINER (0/1) & KONVERSI
# ==================================================
def detect_binary_labels(df, non_labels):
    cand = [c for c in df.columns if c not in non_labels]
    label_cols, skipped = [], []
    map_bool = {"true":1, "false":0, "y":1, "n":0, "yes":1, "no":0}
    for c in cand:
        s = df[c].dropna().astype(str).str.lower()
        uniq = set(s.unique())
        if uniq <= {"0", "1"}:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
            label_cols.append(c)
        elif uniq <= set(list(map_bool.keys()) + ["0","1"]):
            df[c] = (
                df[c].astype(str).str.lower().map(map_bool)
                .fillna(pd.to_numeric(df[c], errors="coerce"))
                .fillna(0).astype(int)
            )
            label_cols.append(c)
        else:
            skipped.append(c)  # bukan biner → abaikan
    return df, label_cols, skipped

df, LABEL_COLS, SKIPPED_COLS = detect_binary_labels(df, NON_LABELS)
if not LABEL_COLS:
    raise ValueError("Tidak ditemukan kolom label biner 0/1.")
print(f"[OK] Jumlah label biner: {len(LABEL_COLS)}")
print("Contoh 10 label:", LABEL_COLS[:10])
if SKIPPED_COLS:
    print(f"[INFO] Kolom non-biner diabaikan (contoh): {SKIPPED_COLS[:10]}")


[OK] Jumlah label biner: 27
Contoh 10 label: ['hepatobiliary disorders', 'metabolism and nutrition disorders', 'product issues', 'eye disorders', 'investigations', 'musculoskeletal and connective tissue disorders', 'gastrointestinal disorders', 'social circumstances', 'immune system disorders', 'reproductive system and breast disorders']


In [7]:
# =================================================
# STEP 3 — GROUPS via Murcko scaffold (anti-leak)
# =================================================
def smiles_to_group(smi: str):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return smi  # fallback
    try:
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=m)
        return scaf if scaf else Chem.MolToSmiles(m)
    except Exception:
        return Chem.MolToSmiles(m) if m else smi

df["group_id"] = df["smiles"].astype(str).apply(smiles_to_group)
GROUPS = df["group_id"].astype(str).values
print("[OK] GROUPS siap. Contoh:", df["group_id"].head().tolist())

[OK] GROUPS siap. Contoh: ['NCCNCCNCCNCCN', 'CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c2=O)cc1O', 'C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]3C(=C)C[C@@]21CC', 'C#C[C@]1(O)CCC2C3CCC4=CC(=O)CCC4C3C(=C)CC21CC', 'NC(=O)N1c2ccccc2CC(O)c2ccccc21']


In [8]:
# ========================================================
# STEP 4 — MORGAN 2048 bit → CSR uint8 (0/1), simpan NPZ
# ========================================================
from scipy.sparse import csr_matrix, save_npz

# Tentukan nama file output NPZ
OUT_X_NPZ_INT = "X_features_morgan_2048.npz"

def morgan_to_csr_uint8(smiles_list, n_bits=2048, radius=2,
                        use_chirality=True, use_bond_types=True, use_features=False):
    data, indices, indptr = [], [], [0]
    bad_idx = []
    for i, smi in enumerate(smiles_list):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            bad_idx.append(i); indptr.append(len(indices)); continue
        bv = AllChem.GetMorganFingerprintAsBitVect(
            m, radius=radius, nBits=n_bits,
            useChirality=use_chirality, useBondTypes=use_bond_types, useFeatures=use_features
        )
        onbits = bv.GetOnBits()
        indices.extend(onbits)
        data.extend([1]*len(onbits))  # integer 1
        indptr.append(len(indices))
    X_uint8 = csr_matrix((np.asarray(data, dtype=np.uint8),
                          np.asarray(indices, dtype=np.int32),
                          np.asarray(indptr, dtype=np.int32)),
                         shape=(len(smiles_list), n_bits), dtype=np.uint8)
    return X_uint8, bad_idx

smiles_list = df["smiles"].astype(str).tolist()
X_INT, bad_rows = morgan_to_csr_uint8(
    smiles_list, n_bits=MORGAN_BITS, radius=MORGAN_RADIUS,
    use_chirality=True, use_bond_types=True, use_features=False
)

assert X_INT.shape[1] == MORGAN_BITS, f"Panjang vektor = {X_INT.shape[1]}, harus {MORGAN_BITS}."
nnz = X_INT.nnz
tot = X_INT.shape[0] * X_INT.shape[1]
print(f"[OK] X_INT shape={X_INT.shape}, dtype={X_INT.dtype}, sparsity={1 - nnz/tot:.6f}")
if bad_rows:
    print(f"[WARN] {len(bad_rows)} SMILES invalid → baris nol. Contoh idx: {bad_rows[:10]}")

save_npz(OUT_X_NPZ_INT, X_INT)
print(f"[SAVE] NPZ fitur: {OUT_X_NPZ_INT}")

[OK] X_INT shape=(1427, 2048), dtype=uint8, sparsity=0.977223
[SAVE] NPZ fitur: X_features_morgan_2048.npz


In [9]:
# ==========================================================
# STEP 5 — ANALISIS IR & PILIH 3 LABEL (besar/median/kecil)
# ==========================================================
OUT_IMB_CSV = "imbalance_summary.csv"

def ir_ratio(neg, pos):
    maj, mino = (neg, pos) if neg >= pos else (pos, neg)
    return float("inf") if mino == 0 else round(maj/mino, 6)

rows = []
n = len(df)
for c in LABEL_COLS:
    pos = int(df[c].sum()); neg = n - pos
    rows.append({"Label": c, "Negative":neg, "Positive":pos, "IR(maj/min)": ir_ratio(neg,pos), "Pos%": round(pos/n,4)})
imb = pd.DataFrame(rows).sort_values("IR(maj/min)", ascending=False).reset_index(drop=True)
imb.to_csv(OUT_IMB_CSV, index=False)
print(f"[SAVE] Ringkasan imbalance: {OUT_IMB_CSV}")

m = len(imb)
largest  = imb.iloc[0]["Label"]
median   = imb.iloc[m//2]["Label"]
smallest = imb.iloc[-1]["Label"]
SELECTED_LABELS = list(dict.fromkeys([largest, median, smallest]))  # unik, pertahankan urutan
print("[OK] 3 label terpilih (IR):", SELECTED_LABELS)

[SAVE] Ringkasan imbalance: imbalance_summary.csv
[OK] 3 label terpilih (IR): ['product issues', 'neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'reproductive system and breast disorders']


In [10]:
# ===================================================
# STEP 6 — GABUNGKAN smiles + bit_0..bit_2047 + 3 label
# ===================================================
# Buat DataFrame sparse untuk fitur (0/1 integer)
OUT_COMBINED_CSV = "combined_dataset.csv"

bit_cols = [f"bit_{j}" for j in range(MORGAN_BITS)]
X_df = pd.DataFrame.sparse.from_spmatrix(X_INT, columns=bit_cols).astype(pd.SparseDtype("uint8"))

# Identitas (hanya 'smiles')
id_df = df[["smiles"]].reset_index(drop=True)

# Label terpilih
labels_df = df[SELECTED_LABELS].astype(int).reset_index(drop=True)

# Gabungkan
combined_df = pd.concat([id_df, X_df.reset_index(drop=True), labels_df], axis=1)

# Sanity: pastikan 2048 kolom bit
bit_cols_in_df = [c for c in combined_df.columns if c.startswith("bit_")]
assert len(bit_cols_in_df) == MORGAN_BITS, f"Kolom bit = {len(bit_cols_in_df)}, harus {MORGAN_BITS}."

combined_df.to_csv(OUT_COMBINED_CSV, index=False)

In [11]:
# =======================================
# TAMPILKAN DATAFRAME: smiles + fitur + 3 organ
# =======================================
import pandas as pd

def build_combined_df_if_needed():
    if "combined_df" in globals():
        return combined_df

    # Guard minimal
    assert "df" in globals(), "`df` belum ada. Jalankan STEP 1 (load dari Google Sheets)."
    assert "X_INT" in globals(), "`X_INT` belum ada. Jalankan STEP 4 (Morgan 2048)."
    assert "SELECTED_LABELS" in globals() and len(SELECTED_LABELS) == 3, \
        "`SELECTED_LABELS` belum ada / tidak 3 label. Jalankan STEP 5."

    # Buat kolom fitur bit_0..bit_{n-1} (0/1 integer, sparse)
    n_bits = X_INT.shape[1]
    bit_cols = [f"bit_{j}" for j in range(n_bits)]
    X_df = pd.DataFrame.sparse.from_spmatrix(X_INT, columns=bit_cols).astype(pd.SparseDtype("uint8"))

    # Gabungkan: smiles + fitur + 3 label
    id_df     = df[["smiles"]].reset_index(drop=True)
    labels_df = df[SELECTED_LABELS].astype(int).reset_index(drop=True)
    combined  = pd.concat([id_df, X_df.reset_index(drop=True), labels_df], axis=1)

    # Sanity
    bit_cols_in_df = [c for c in combined.columns if c.startswith("bit_")]
    assert len(bit_cols_in_df) == n_bits, f"Jumlah kolom bit di DataFrame = {len(bit_cols_in_df)} != {n_bits}"

    globals()["combined_df"] = combined
    return combined

combined_df = build_combined_df_if_needed()

# ==== Tampilkan ====
print("=== SHAPE (rows, cols) ===", combined_df.shape)
print("=== Kolom pertama ===", combined_df.columns[:10].tolist(), "...")
print("=== Kolom label (3 organ) ===", list(SELECTED_LABELS))

try:
    from IPython.display import display
    # 1) Preview penuh (5 baris pertama)
    display(combined_df.head(5))
    # 2) Preview ringkas: hanya smiles + 3 organ
    #display(combined_df[["smiles", *SELECTED_LABELS]].head(10))
except Exception:
    # Fallback terminal
    print("\n--- Preview (5 baris) ---")
    print(combined_df.head(5).to_string(index=False))
    #print("\n--- Hanya smiles + 3 organ (10 baris) ---")
    #print(combined_df[["smiles", *SELECTED_LABELS]].head(10).to_string(index=False))

=== SHAPE (rows, cols) === (1427, 2052)
=== Kolom pertama === ['smiles', 'bit_0', 'bit_1', 'bit_2', 'bit_3', 'bit_4', 'bit_5', 'bit_6', 'bit_7', 'bit_8'] ...
=== Kolom label (3 organ) === ['product issues', 'neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'reproductive system and breast disorders']


,smiles,bit_0,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,...,bit_2041,bit_2042,bit_2043,bit_2044,bit_2045,bit_2046,bit_2047,product issues,"neoplasms benign, malignant and unspecified (incl cysts and polyps)",reproductive system and breast disorders
0,C(CNCCNCCNCCN)N ...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CC(C)(C)C1=CC(=C(C=C1NC(=O)C2=CNC3=CC=CC=C3C2=...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,CC[C@]12CC(=C)[C@H]3[C@H]([C@@H]1CC[C@]2(C#C)O...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
3,CCC12CC(=C)C3C(C1CC[C@]2(C#C)O)CCC4=CC(=O)CCC3...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
4,C1C(C2=CC=CC=C2N(C3=CC=CC=C31)C(=O)N)O ...,1,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [12]:
# =======================================
# STEP 7 — BASELINE TANPA RESAMPLING (TOP-2, REVISI)
# Model: LinearSVC & RidgeClassifier
# Group-CV, metrik: F1 / BAS / GM
# =======================================
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import RidgeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score, confusion_matrix

# ---------- Guards ----------
assert "X_INT" in globals(), "X_INT belum ada (jalankan STEP 4)."
assert "df" in globals() and "GROUPS" in globals(), "df/GROUPS belum ada (jalankan STEP 1 & 3)."
assert "SELECTED_LABELS" in globals() and len(SELECTED_LABELS) == 3, "SELECTED_LABELS harus 3 label (STEP 5)."
try:
    RANDOM_STATE
except NameError:
    RANDOM_STATE = 42

# ---------- Fitur untuk model (float32) ----------
X = X_INT.astype(np.float32)

# ---------- Metrik GM (Geometric Mean) ----------
def gm_score(y_true, y_pred):
    # labels=[0,1] memastikan matriks konfusi bentuk 2x2 meski ada kelas yg kosong
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # sensitivity
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0  # specificity
    return float(np.sqrt(tpr * tnr))

# scorer dengan zero_division=0 agar aman saat tidak ada prediksi positif
scoring = {
    "f1":  make_scorer(f1_score, zero_division=0),
    "bas": make_scorer(balanced_accuracy_score),
    "gm":  make_scorer(gm_score),
}

# ---------- CV splitter (group-aware) ----------
def make_group_cv(y, groups, n_splits=5, seed0=RANDOM_STATE, max_tries=50):
    """
    Kembalikan StratifiedGroupKFold yang valid: tiap fold train & valid punya ≥1 positif.
    Jika grup-positif terlalu sedikit, otomatis mengurangi n_splits (minimal 2).
    """
    pos_groups = set(g for g, yy in zip(groups, y) if yy == 1)
    if len(pos_groups) < n_splits:
        n_splits = max(2, len(pos_groups))
    for sd in range(seed0, seed0 + max_tries):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=sd)
        ok = True
        for tr, va in cv.split(np.zeros(len(y)), y, groups):
            if (y[tr].sum() == 0) or (y[va].sum() == 0):
                ok = False
                break
        if ok:
            return cv
    # fallback aman
    return StratifiedGroupKFold(n_splits=max(2, min(3, n_splits)), shuffle=True, random_state=seed0)

# ---------- Pipelines TOP-2 ----------
def pipe_linearsvc(C=1.0):
    return Pipeline([
        ("scaler", StandardScaler(with_mean=False)),  # aman untuk sparse
        ("clf", LinearSVC(C=float(C),
                          class_weight="balanced",
                          max_iter=10000,
                          random_state=RANDOM_STATE)),
    ])

def pipe_ridge(alpha=1.0):
    return Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", RidgeClassifier(alpha=float(alpha),
                                class_weight="balanced",
                                random_state=RANDOM_STATE)),
    ])

MODELS = [
    ("LinearSVC", pipe_linearsvc(C=1.0)),
    ("Ridge",     pipe_ridge(alpha=1.0)),
]

# ---------- Evaluasi: tiap label × 2 model ----------
rows = []
for lab in SELECTED_LABELS:
    y = df[lab].astype(int).values
    cv = make_group_cv(y, GROUPS, n_splits=5, seed0=RANDOM_STATE)

    for name, pipe in MODELS:
        cvres = cross_validate(
            pipe, X, y, groups=GROUPS, cv=cv,
            scoring=scoring, n_jobs=-1, return_train_score=False
        )
        rows.append({
            "Label": lab,
            "Model": name,
            "F1_mean":  float(np.mean(cvres["test_f1"])),
            "F1_std":   float(np.std(cvres["test_f1"])),
            "BAS_mean": float(np.mean(cvres["test_bas"])),
            "BAS_std":  float(np.std(cvres["test_bas"])),
            "GM_mean":  float(np.mean(cvres["test_gm"])),
            "GM_std":   float(np.std(cvres["test_gm"]))
        })
        print(f"[BASELINE][{lab}][{name}] "
              f"F1={rows[-1]['F1_mean']:.4f}±{rows[-1]['F1_std']:.4f} | "
              f"BAS={rows[-1]['BAS_mean']:.4f}±{rows[-1]['BAS_std']:.4f} | "
              f"GM={rows[-1]['GM_mean']:.4f}±{rows[-1]['GM_std']:.4f}")

res_top2 = pd.DataFrame(rows)

# ---------- Simpan CSV ----------
_ts = globals().get("_ts", datetime.now().strftime("%Y%m%d-%H%M%S"))
out_csv = f"baseline_no_resampling_top2_{_ts}.csv"
res_top2.to_csv(out_csv, index=False)
print(f"[STEP7] Hasil baseline (top-2) disimpan: {out_csv}")

# ---------- TAMPILKAN: masing-masing label menampilkan 2 baris (Top-2 models) ----------
def _fmt(ms, ss): return f"{ms:.4f} ± {ss:.4f}"

def show_results_per_label(res_df, label_list):
    for lab in label_list:
        sub = res_df[res_df["Label"] == lab].copy()
        # urutkan dari GM terbaik
        sub = sub.sort_values("GM_mean", ascending=False)
        # format mean±std untuk tampilan
        sub["F1"]  = [_fmt(m, s) for m, s in zip(sub["F1_mean"],  sub["F1_std"])]
        sub["BAS"] = [_fmt(m, s) for m, s in zip(sub["BAS_mean"], sub["BAS_std"])]
        sub["GM"]  = [_fmt(m, s) for m, s in zip(sub["GM_mean"],  sub["GM_std"])]
        view = sub[["Model", "F1", "BAS", "GM"]].reset_index(drop=True)

        print(f"\n=== HASIL PER LABEL: {lab} ===")
        try:
            from IPython.display import display
            display(view)
        except Exception:
            print(view.to_string(index=False))

show_results_per_label(res_top2, SELECTED_LABELS)

# ---------- (Opsional) Ringkasan terbaik per label (berdasarkan GM_mean) ----------
best_by_gm = res_top2.sort_values(["Label","GM_mean"], ascending=[True, False])\
                     .groupby("Label").head(1)[["Label","Model","GM_mean","F1_mean","BAS_mean"]]
best_by_gm = best_by_gm.rename(columns={
    "GM_mean":"GM_best", "F1_mean":"F1_at_bestGM", "BAS_mean":"BAS_at_bestGM"
}).reset_index(drop=True)
print("\n=== RINGKASAN TERBAIK (berdasarkan GM) ===")
try:
    from IPython.display import display
    display(best_by_gm)
except Exception:
    print(best_by_gm.to_string(index=False))

[BASELINE][product issues][LinearSVC] F1=0.0000±0.0000 | BAS=0.4936±0.0014 | GM=0.0000±0.0000
[BASELINE][product issues][Ridge] F1=0.0364±0.0727 | BAS=0.5126±0.0510 | GM=0.0989±0.1979
[BASELINE][neoplasms benign, malignant and unspecified (incl cysts and polyps)][LinearSVC] F1=0.4566±0.0468 | BAS=0.6335±0.0218 | GM=0.6072±0.0321
[BASELINE][neoplasms benign, malignant and unspecified (incl cysts and polyps)][Ridge] F1=0.4236±0.0446 | BAS=0.6052±0.0290 | GM=0.5872±0.0369
[BASELINE][reproductive system and breast disorders][LinearSVC] F1=0.6452±0.0231 | BAS=0.6434±0.0170 | GM=0.6431±0.0168
[BASELINE][reproductive system and breast disorders][Ridge] F1=0.6136±0.0226 | BAS=0.6148±0.0130 | GM=0.6143±0.0136
[STEP7] Hasil baseline (top-2) disimpan: baseline_no_resampling_top2_20260512-101019.csv

=== HASIL PER LABEL: product issues ===


,Model,F1,BAS,GM
0,Ridge,0.0364 ± 0.0727,0.5126 ± 0.0510,0.0989 ± 0.1979
1,LinearSVC,0.0000 ± 0.0000,0.4936 ± 0.0014,0.0000 ± 0.0000



=== HASIL PER LABEL: neoplasms benign, malignant and unspecified (incl cysts and polyps) ===


,Model,F1,BAS,GM
0,LinearSVC,0.4566 ± 0.0468,0.6335 ± 0.0218,0.6072 ± 0.0321
1,Ridge,0.4236 ± 0.0446,0.6052 ± 0.0290,0.5872 ± 0.0369



=== HASIL PER LABEL: reproductive system and breast disorders ===


,Model,F1,BAS,GM
0,LinearSVC,0.6452 ± 0.0231,0.6434 ± 0.0170,0.6431 ± 0.0168
1,Ridge,0.6136 ± 0.0226,0.6148 ± 0.0130,0.6143 ± 0.0136



=== RINGKASAN TERBAIK (berdasarkan GM) ===


,Label,Model,GM_best,F1_at_bestGM,BAS_at_bestGM
0,"neoplasms benign, malignant and unspecified (i...",LinearSVC,0.607239,0.456629,0.633478
1,product issues,Ridge,0.098927,0.036364,0.512562
2,reproductive system and breast disorders,LinearSVC,0.643120,0.645206,0.643375


In [13]:
# # =======================================
# # BAR CHART: Baseline Top-2 (LinearSVC & Ridge) — GM, F1, BAS
# # =======================================
# import os, pandas as pd, numpy as np, matplotlib.pyplot as plt
# from datetime import datetime

# # 1) Ambil data hasil STEP 7
# # Jika res_top2 sudah ada di memori, dipakai langsung.
# # Jika belum, isi path CSV kamu di sini (lihat path yang dicetak STEP 7):
# BASELINE_TOP2_CSV = None  # contoh: "/content/baseline_no_resampling_top2_20250907-120301.csv"

# if "res_top2" in globals() and isinstance(res_top2, pd.DataFrame):
#     res = res_top2.copy()
# elif BASELINE_TOP2_CSV and os.path.exists(BASELINE_TOP2_CSV):
#     res = pd.read_csv(BASELINE_TOP2_CSV)
# else:
#     raise RuntimeError(
#         "Tidak menemukan `res_top2` di memori dan BASELINE_TOP2_CSV belum diisi/berkasnya tidak ada.\n"
#         "Solusi: 1) Jalankan STEP 7 dulu, atau 2) isi variabel BASELINE_TOP2_CSV dengan path CSV hasil STEP 7."
#     )

# # 2) Pastikan hanya 2 model per label (LinearSVC & Ridge); kalau lebih, ambil top-2 by GM
# res = res.sort_values(["Label","GM_mean"], ascending=[True, False])\
#          .groupby("Label", as_index=False, group_keys=False).apply(lambda d: d.head(2)).reset_index(drop=True)

# labels = res["Label"].unique().tolist()
# preferred_order = ["LinearSVC", "Ridge"]
# models = [m for m in preferred_order if m in res["Model"].unique().tolist()]
# for m in res["Model"].unique():
#     if m not in models:
#         models.append(m)

# def wide_metric(df, mean_col, std_col):
#     m = df.pivot(index="Label", columns="Model", values=mean_col).reindex(labels)[models]
#     s = df.pivot(index="Label", columns="Model", values=std_col).reindex(labels)[models]
#     return m, s

# gm_m, gm_s   = wide_metric(res, "GM_mean",  "GM_std")
# f1_m, f1_s   = wide_metric(res, "F1_mean",  "F1_std")
# bas_m, bas_s = wide_metric(res, "BAS_mean", "BAS_std")

# def plot_grouped_bars(means_df, stds_df, title, ylabel, filename):
#     fig, ax = plt.subplots(figsize=(10, 5))
#     x = np.arange(len(means_df.index))
#     n_models = len(means_df.columns)
#     width = 0.8 / max(1, n_models)

#     for i, model in enumerate(means_df.columns):
#         offsets = x + (i - (n_models-1)/2) * width
#         vals = means_df[model].values
#         errs = stds_df[model].values
#         ax.bar(offsets, vals, width, yerr=errs, capsize=4, label=model)
#         for xo, yo in zip(offsets, vals):
#             ax.text(xo, yo + (np.nanmax(vals)*0.01 if np.isfinite(yo) else 0.01),
#                     f"{yo:.3f}", ha="center", va="bottom", fontsize=8)

#     ax.set_title(title)
#     ax.set_ylabel(ylabel)
#     ax.set_xticks(x)
#     ax.set_xticklabels(means_df.index, rotation=15, ha="right")
#     ax.legend()
#     ax.set_ylim(bottom=0.0)
#     fig.tight_layout()
#     out_path = filename
#     fig.savefig(out_path, dpi=150, bbox_inches="tight")
#     plt.show()
#     print("Saved:", out_path)

# timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
# plot_grouped_bars(gm_m,  gm_s,  "Baseline (No Resampling) — GM by Model & Label",  "GM (mean ± std)",  f"baseline_top2_GM_{timestamp}.png")
# plot_grouped_bars(f1_m,  f1_s,  "Baseline (No Resampling) — F1 by Model & Label",  "F1 (mean ± std)",  f"baseline_top2_F1_{timestamp}.png")
# plot_grouped_bars(bas_m, bas_s, "Baseline (No Resampling) — BAS by Model & Label", "BAS (mean ± std)", f"baseline_top2_BAS_{timestamp}.png")

# # 3) Tabel ringkasan rapi
# def fmt(m,s): return f"{m:.4f} ± {s:.4f}"
# summary = res.copy()
# summary["F1"]  = [fmt(m,s) for m,s in zip(summary["F1_mean"],  summary["F1_std"])]
# summary["BAS"] = [fmt(m,s) for m,s in zip(summary["BAS_mean"], summary["BAS_std"])]
# summary["GM"]  = [fmt(m,s) for m,s in zip(summary["GM_mean"],  summary["GM_std"])]
# summary = summary[["Label","Model","F1","BAS","GM"]].sort_values(["Label","GM"], ascending=[True, False])
# summary

In [14]:
# =======================================
# STEP 8 — OVERSAMPLING-ONLY (SMOTE & ADASYN)
# Minority dinaikkan > mayoritas; evaluasi Group-CV (tanpa undersampling)
# =======================================
import numpy as np
import pandas as pd
from collections import Counter
from datetime import datetime

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix

from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import TomekLinks  # (opsional; default OFF)

# -------- Guards --------
assert "X_INT" in globals(), "X_INT belum ada (jalankan STEP 4)."
assert "df" in globals() and "GROUPS" in globals(), "df/GROUPS belum ada (STEP 1 & 3)."
assert "SELECTED_LABELS" in globals() and len(SELECTED_LABELS) == 3, "SELECTED_LABELS harus 3 label (STEP 5)."
try:
    RANDOM_STATE
except NameError:
    RANDOM_STATE = 42

# -------- Konfigurasi utama --------
OVER_METHODS = ["SMOTE", "ADASYN"]          # metode oversampling yang diuji
MINORITY_MULTIPLIER = 1.30                  # >> 1.0 → minority_after ≈ 1.3 × majority
APPLY_TOMEK = False                         # kalau mau, set True (pembersihan setelah oversampling)
N_SPLITS = 5                                # outer CV splits

# Dua model terbaik untuk evaluasi
def make_model(name):
    if name == "LinearSVC":
        # tanpaclass_weight agar efek oversampling terlihat jelas
        return LinearSVC(C=1.0, class_weight=None, max_iter=10000, random_state=RANDOM_STATE)
    if name == "Ridge":
        return RidgeClassifier(alpha=1.0, class_weight=None, random_state=RANDOM_STATE)
    raise ValueError("Unknown model name")

TOP2_MODELS = ["LinearSVC", "Ridge"]

# -------- Metrik --------
def gm_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    tpr = tp/(tp+fn) if (tp+fn)>0 else 0.0
    tnr = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return float(np.sqrt(tpr*tnr))

# -------- CV splitter (group-aware, memastikan ada positif) --------
def make_group_cv(y, groups, n_splits=N_SPLITS, seed0=RANDOM_STATE, max_tries=50):
    pos_groups = set(g for g, yy in zip(groups, y) if yy==1)
    if len(pos_groups) < n_splits:
        n_splits = max(2, len(pos_groups))
    for sd in range(seed0, seed0+max_tries):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=sd)
        ok = True
        for tr, va in cv.split(np.zeros_like(y), y, groups):
            if (y[tr].sum()==0) or (y[va].sum()==0):
                ok = False; break
        if ok: return cv
    return StratifiedGroupKFold(n_splits=max(2, min(3, n_splits)), shuffle=True, random_state=seed0)

# -------- Oversampling per fold (dinamis, > mayoritas) --------
def oversample_train(X_tr_csr, y_tr, method: str, multiplier: float, use_tomek: bool):
    """
    - Hitung kelas mayoritas/minoritas pada TRAIN fold
    - Standarisasi (fit di TRAIN)
    - Oversample minority hingga target > mayoritas (via dict sampling_strategy)
    - (Opsional) TomekLinks setelah OS
    - Kembalikan: X_os_scaled, y_os, scaler, info_count_before/after
    """
    cnt = Counter(y_tr)
    # kalau 1 kelas: tidak bisa OS algoritmik → skip
    if len(cnt) < 2:
        scaler = StandardScaler(with_mean=True, with_std=True)
        Xd = X_tr_csr.toarray() if hasattr(X_tr_csr, "toarray") else np.asarray(X_tr_csr)
        Xs = scaler.fit_transform(Xd)
        return Xs, y_tr, scaler, {"before": cnt, "after": cnt}

    # identifikasi majority/minority
    maj = max(cnt, key=cnt.get)
    minc = min(cnt, key=cnt.get)
    n_maj = cnt[maj]
    n_min = cnt[minc]

    # target minority > majority
    target_min = int(np.ceil(multiplier * n_maj))

    # siapkan data dense & skalakan untuk tetangga
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xd = X_tr_csr.toarray() if hasattr(X_tr_csr, "toarray") else np.asarray(X_tr_csr)
    Xs = scaler.fit_transform(Xd)

    # tentukan k tetangga aman (SMOTE/ADASYN butuh >=1)
    k = max(1, min(5, n_min - 1))

    # fallback: kalau n_min < 2, pakai ROS agar tetap bisa “naikkan”
    can_algo = (n_min >= 2)

    if method.upper() == "SMOTE" and can_algo:
        sampler = SMOTE(sampling_strategy={minc: target_min}, k_neighbors=k, random_state=RANDOM_STATE)
        X_os, y_os = sampler.fit_resample(Xs, y_tr)
    elif method.upper() == "ADASYN" and can_algo:
        sampler = ADASYN(sampling_strategy={minc: target_min}, n_neighbors=k, random_state=RANDOM_STATE)
        X_os, y_os = sampler.fit_resample(Xs, y_tr)
    else:
        # fallback: duplikasi sederhana (tetap > mayoritas)
        ros = RandomOverSampler(sampling_strategy={minc: target_min}, random_state=RANDOM_STATE)
        X_os, y_os = ros.fit_resample(Xs, y_tr)

    if use_tomek:
        tl = TomekLinks(n_jobs=-1)
        X_os, y_os = tl.fit_resample(X_os, y_os)

    after_cnt = Counter(y_os)
    info = {"before": cnt, "after": after_cnt, "k_used": k}
    return X_os, y_os, scaler, info

# -------- Evaluasi utama: per label × (SMOTE, ADASYN) × (LinearSVC, Ridge) --------
rows = []
for lab in SELECTED_LABELS:
    y_all = df[lab].astype(int).values
    cv = make_group_cv(y_all, GROUPS, n_splits=N_SPLITS, seed0=RANDOM_STATE)

    for over in OVER_METHODS:
        for model_name in TOP2_MODELS:
            f1s, bass, gms = [], [], []
            fold_logs = []

            for fold_id, (tr, va) in enumerate(cv.split(np.zeros_like(y_all), y_all, GROUPS), start=1):
                X_tr, X_va = X_INT[tr], X_INT[va]
                y_tr, y_va = y_all[tr], y_all[va]

                # 1) Oversample TRAIN saja (minority > majority)
                X_os, y_os, scaler, info = oversample_train(
                    X_tr, y_tr, method=over, multiplier=MINORITY_MULTIPLIER, use_tomek=APPLY_TOMEK
                )

                # 2) Transform VALID pakai scaler TRAIN
                X_va_scaled = scaler.transform(X_va.toarray() if hasattr(X_va, "toarray") else np.asarray(X_va))

                # 3) Train classifier pada data oversampled (tanpa scaler lagi; sudah scaled)
                clf = make_model(model_name)
                clf.fit(X_os, y_os)

                # 4) Prediksi & metrik
                y_pred = clf.predict(X_va_scaled)
                f1s.append(f1_score(y_va, y_pred, zero_division=0))
                bass.append(balanced_accuracy_score(y_va, y_pred))
                gms.append(gm_score(y_va, y_pred))

                # log per fold (opsional untuk audit)
                fold_logs.append({
                    "Fold": fold_id,
                    "Before_pos": int(info["before"].get(1, 0)),
                    "Before_neg": int(info["before"].get(0, 0)),
                    "After_pos":  int(info["after"].get(1, 0)),
                    "After_neg":  int(info["after"].get(0, 0)),
                    "k_neighbors": info["k_used"],
                })

            # ringkasan per kombinasi
            rows.append({
                "Label": lab,
                "Method_Over": over,
                "Model": model_name,
                "MinorityMultiplier": MINORITY_MULTIPLIER,
                "APPLY_TOMEK": APPLY_TOMEK,
                "F1_mean":  float(np.mean(f1s)),  "F1_std":  float(np.std(f1s)),
                "BAS_mean": float(np.mean(bass)), "BAS_std": float(np.std(bass)),
                "GM_mean":  float(np.mean(gms)),  "GM_std":  float(np.std(gms)),
            })

            # cetak ringkasan + contoh fold 1
            ex = fold_logs[0]
            print(f"[OVER][{lab}][{over}][{model_name}] "
                  f"F1={rows[-1]['F1_mean']:.4f}±{rows[-1]['F1_std']:.4f} | "
                  f"BAS={rows[-1]['BAS_mean']:.4f}±{rows[-1]['BAS_std']:.4f} | "
                  f"GM={rows[-1]['GM_mean']:.4f}±{rows[-1]['GM_std']:.4f} "
                  f"| Fold1 Before(+/−)={ex['Before_pos']}/{ex['Before_neg']} → After(+/−)={ex['After_pos']}/{ex['After_neg']}")

res_over_only = pd.DataFrame(rows)

# -------- Simpan hasil --------
_ts = globals().get("_ts", datetime.now().strftime("%Y%m%d-%H%M%S"))
out_csv = f"oversampling_only_SMOTE_ADASYN_top2_{_ts}.csv"
res_over_only.to_csv(out_csv, index=False)
print(f"[STEP8] Hasil oversampling-only disimpan: {out_csv}")

# -------- Tampilkan ringkasan rapi --------
def fmt(m,s): return f"{m:.4f} ± {s:.4f}"
view = res_over_only.copy()
view["F1"]  = [fmt(m,s) for m,s in zip(view["F1_mean"],  view["F1_std"])]
view["BAS"] = [fmt(m,s) for m,s in zip(view["BAS_mean"], view["BAS_std"])]
view["GM"]  = [fmt(m,s) for m,s in zip(view["GM_mean"],  view["GM_std"])]
view = view[["Label","Method_Over","Model","MinorityMultiplier","APPLY_TOMEK","F1","BAS","GM"]]\
       .sort_values(["Label","Method_Over","GM"], ascending=[True, True, False]).reset_index(drop=True)

try:
    from IPython.display import display
    display(view)
except Exception:
    print(view.to_string(index=False))

[OVER][product issues][SMOTE][LinearSVC] F1=0.0000±0.0000 | BAS=0.4918±0.0021 | GM=0.0000±0.0000 | Fold1 Before(+/−)=17/1124 → After(+/−)=1462/1124
[OVER][product issues][SMOTE][Ridge] F1=0.0364±0.0727 | BAS=0.5126±0.0510 | GM=0.0989±0.1979 | Fold1 Before(+/−)=17/1124 → After(+/−)=1462/1124
[OVER][product issues][ADASYN][LinearSVC] F1=0.0000±0.0000 | BAS=0.4918±0.0021 | GM=0.0000±0.0000 | Fold1 Before(+/−)=17/1124 → After(+/−)=1462/1124
[OVER][product issues][ADASYN][Ridge] F1=0.0364±0.0727 | BAS=0.5126±0.0510 | GM=0.0989±0.1979 | Fold1 Before(+/−)=17/1124 → After(+/−)=1462/1124


/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[OVER][neoplasms benign, malignant and unspecified (incl cysts and polyps)][SMOTE][LinearSVC] F1=0.4505±0.0410 | BAS=0.6270±0.0248 | GM=0.6076±0.0338 | Fold1 Before(+/−)=303/838 → After(+/−)=1090/838
[OVER][neoplasms benign, malignant and unspecified (incl cysts and polyps)][SMOTE][Ridge] F1=0.4188±0.0475 | BAS=0.6014±0.0346 | GM=0.5830±0.0421 | Fold1 Before(+/−)=303/838 → After(+/−)=1090/838


/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  

[OVER][neoplasms benign, malignant and unspecified (incl cysts and polyps)][ADASYN][LinearSVC] F1=0.4505±0.0410 | BAS=0.6270±0.0248 | GM=0.6076±0.0338 | Fold1 Before(+/−)=303/838 → After(+/−)=1084/838
[OVER][neoplasms benign, malignant and unspecified (incl cysts and polyps)][ADASYN][Ridge] F1=0.4207±0.0481 | BAS=0.6027±0.0350 | GM=0.5847±0.0428 | Fold1 Before(+/−)=303/838 → After(+/−)=1084/838


/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[OVER][reproductive system and breast disorders][SMOTE][LinearSVC] F1=0.6382±0.0246 | BAS=0.6412±0.0171 | GM=0.6406±0.0169 | Fold1 Before(+/−)=576/565 → After(+/−)=576/749
[OVER][reproductive system and breast disorders][SMOTE][Ridge] F1=0.6132±0.0218 | BAS=0.6141±0.0125 | GM=0.6135±0.0131 | Fold1 Before(+/−)=576/565 → After(+/−)=576/749


/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  

[OVER][reproductive system and breast disorders][ADASYN][LinearSVC] F1=0.6386±0.0251 | BAS=0.6419±0.0183 | GM=0.6413±0.0180 | Fold1 Before(+/−)=576/565 → After(+/−)=576/767
[OVER][reproductive system and breast disorders][ADASYN][Ridge] F1=0.6132±0.0218 | BAS=0.6141±0.0125 | GM=0.6135±0.0131 | Fold1 Before(+/−)=576/565 → After(+/−)=576/767
[STEP8] Hasil oversampling-only disimpan: oversampling_only_SMOTE_ADASYN_top2_20260512-101019.csv


,Label,Method_Over,Model,MinorityMultiplier,APPLY_TOMEK,F1,BAS,GM
0,"neoplasms benign, malignant and unspecified (i...",ADASYN,LinearSVC,1.3,False,0.4505 ± 0.0410,0.6270 ± 0.0248,0.6076 ± 0.0338
1,"neoplasms benign, malignant and unspecified (i...",ADASYN,Ridge,1.3,False,0.4207 ± 0.0481,0.6027 ± 0.0350,0.5847 ± 0.0428
2,"neoplasms benign, malignant and unspecified (i...",SMOTE,LinearSVC,1.3,False,0.4505 ± 0.0410,0.6270 ± 0.0248,0.6076 ± 0.0338
3,"neoplasms benign, malignant and unspecified (i...",SMOTE,Ridge,1.3,False,0.4188 ± 0.0475,0.6014 ± 0.0346,0.5830 ± 0.0421
4,product issues,ADASYN,Ridge,1.3,False,0.0364 ± 0.0727,0.5126 ± 0.0510,0.0989 ± 0.1979
5,product issues,ADASYN,LinearSVC,1.3,False,0.0000 ± 0.0000,0.4918 ± 0.0021,0.0000 ± 0.0000
6,product issues,SMOTE,Ridge,1.3,False,0.0364 ± 0.0727,0.5126 ± 0.0510,0.0989 ± 0.1979
7,product issues,SMOTE,LinearSVC,1.3,False,0.0000 ± 0.0000,0.4918 ± 0.0021,0.0000 ± 0.0000
8,reproductive system and breast disorders,ADASYN,LinearSVC,1.3,False,0.6386 ± 0.0251,0.6419 ± 0.0183,0.6413 ± 0.0180
9,reproductive system and breast disorders,ADASYN,Ridge,1.3,False,0.6132 ± 0.0218,0.6141 ± 0.0125,0.6135 ± 0.0131


In [15]:
# ====== PILIH KOMBINASI TERBAIK PER LABEL (GM utama, F1 & BAS tie-breaker) ======
import os, glob, pandas as pd

def load_res_over_only():
    if "res_over_only" in globals() and isinstance(res_over_only, pd.DataFrame):
        return res_over_only.copy()
    candidates = sorted(glob.glob("oversampling_only_SMOTE_ADASYN_top2_*.csv")) + \
                 sorted(glob.glob("/mnt/data/oversampling_only_SMOTE_ADASYN_top2_*.csv"))
    if not candidates:
        raise RuntimeError("Hasil oversampling belum ditemukan. Jalankan STEP 8 dulu.")
    return pd.read_csv(candidates[-1])

res = load_res_over_only()

# Urutkan dgn prioritas: GM_mean ↓, F1_mean ↓, BAS_mean ↓
ranked = res.sort_values(
    ["Label","GM_mean","F1_mean","BAS_mean"],
    ascending=[True, False, False, False]
)

# Ambil 1 terbaik per label
best_per_label = ranked.groupby("Label", as_index=False).head(1).reset_index(drop=True)

# Tabel ringkas untuk dilihat
view = best_per_label[["Label","Method_Over","Model","MinorityMultiplier","APPLY_TOMEK",
                       "GM_mean","F1_mean","BAS_mean"]].copy()
view["GM_mean"]  = view["GM_mean"].round(4)
view["F1_mean"]  = view["F1_mean"].round(4)
view["BAS_mean"] = view["BAS_mean"].round(4)

print("=== Kombinasi Terbaik per Label (berdasarkan GM) ===")
try:
    from IPython.display import display
    display(view)
except Exception:
    print(view.to_string(index=False))

# (Opsional) juga tampilkan TOP-3 per label untuk perbandingan cepat
top3 = ranked.groupby("Label", as_index=False, group_keys=False).head(3)
top3_view = top3[["Label","Method_Over","Model","GM_mean","F1_mean","BAS_mean"]].copy().round(4)
print("\n=== TOP-3 per Label (GM utama) ===")
try:
    from IPython.display import display
    display(top3_view)
except Exception:
    print(top3_view.to_string(index=False))

# (Opsional) kamus pemenang → untuk dipakai di langkah undersampling refinement berikutnya
WINNERS = {
    row["Label"]: {
        "over": row["Method_Over"],
        "model": row["Model"],
        "multiplier": float(row["MinorityMultiplier"]),
        "use_tomek": bool(row["APPLY_TOMEK"])
    }
    for _, row in best_per_label.iterrows()
}
print("\nWINNERS mapping siap dipakai:", WINNERS)

=== Kombinasi Terbaik per Label (berdasarkan GM) ===


,Label,Method_Over,Model,MinorityMultiplier,APPLY_TOMEK,GM_mean,F1_mean,BAS_mean
0,"neoplasms benign, malignant and unspecified (i...",SMOTE,LinearSVC,1.3,False,0.6076,0.4505,0.6270
1,product issues,SMOTE,Ridge,1.3,False,0.0989,0.0364,0.5126
2,reproductive system and breast disorders,ADASYN,LinearSVC,1.3,False,0.6413,0.6386,0.6419



=== TOP-3 per Label (GM utama) ===


,Label,Method_Over,Model,GM_mean,F1_mean,BAS_mean
4,"neoplasms benign, malignant and unspecified (i...",SMOTE,LinearSVC,0.6076,0.4505,0.6270
6,"neoplasms benign, malignant and unspecified (i...",ADASYN,LinearSVC,0.6076,0.4505,0.6270
7,"neoplasms benign, malignant and unspecified (i...",ADASYN,Ridge,0.5847,0.4207,0.6027
1,product issues,SMOTE,Ridge,0.0989,0.0364,0.5126
3,product issues,ADASYN,Ridge,0.0989,0.0364,0.5126
0,product issues,SMOTE,LinearSVC,0.0000,0.0000,0.4918
10,reproductive system and breast disorders,ADASYN,LinearSVC,0.6413,0.6386,0.6419
8,reproductive system and breast disorders,SMOTE,LinearSVC,0.6406,0.6382,0.6412
9,reproductive system and breast disorders,SMOTE,Ridge,0.6135,0.6132,0.6141



WINNERS mapping siap dipakai: {'neoplasms benign, malignant and unspecified (incl cysts and polyps)': {'over': 'SMOTE', 'model': 'LinearSVC', 'multiplier': 1.3, 'use_tomek': False}, 'product issues': {'over': 'SMOTE', 'model': 'Ridge', 'multiplier': 1.3, 'use_tomek': False}, 'reproductive system and breast disorders': {'over': 'ADASYN', 'model': 'LinearSVC', 'multiplier': 1.3, 'use_tomek': False}}


In [ ]:
# =======================================
# STEP 9 — HHO Undersampling Refinement
# (Outer Group-CV; Inner Group-CV utk fitness)
# =======================================
import numpy as np, pandas as pd
from collections import Counter
from dataclasses import dataclass
from typing import Dict, Tuple, List

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix

from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import TomekLinks

# -------- Guards --------
assert "X_INT" in globals(), "X_INT belum ada (jalankan STEP 4)."
assert "df" in globals() and "GROUPS" in globals(), "df/GROUPS belum ada (STEP 1 & 3)."
assert "SELECTED_LABELS" in globals() and len(SELECTED_LABELS) == 3, "SELECTED_LABELS harus 3 label (STEP 5)."
try:
    RANDOM_STATE
except NameError:
    RANDOM_STATE = 42

# -------- Pemenang oversampling × model per label (pakai hasilmu) --------
# Kamu bisa ubah per label bila perlu.
WINNERS = globals().get("WINNERS", {
    "Product issues": {
        "over": "ADASYN", "model": "LinearSVC", "multiplier": 1.30, "use_tomek": False
    },
    "Reproductive system and breast disorders": {
        "over": "ADASYN", "model": "LinearSVC", "multiplier": 1.30, "use_tomek": False
    },
    "Respiratory, thoracic and mediastinal disorders": {
        "over": "ADASYN", "model": "LinearSVC", "multiplier": 1.30, "use_tomek": False
    },
})

# -------- Metrik utama --------
def gm_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    tpr = tp/(tp+fn) if (tp+fn)>0 else 0.0
    tnr = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return float(np.sqrt(tpr*tnr))

def score_triple(y_true, y_pred) -> Tuple[float,float,float]:
    return (
        f1_score(y_true, y_pred, zero_division=0),
        balanced_accuracy_score(y_true, y_pred),
        gm_score(y_true, y_pred),
    )

# -------- CV helpers --------
def make_group_cv(y, groups, n_splits=5, seed0=RANDOM_STATE, max_tries=50):
    pos_groups = set(g for g, yy in zip(groups, y) if yy==1)
    if len(pos_groups) < n_splits:
        n_splits = max(2, len(pos_groups))
    for sd in range(seed0, seed0+max_tries):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=sd)
        ok=True
        for tr, va in cv.split(np.zeros_like(y), y, groups):
            if (y[tr].sum()==0) or (y[va].sum()==0):
                ok=False; break
        if ok: return cv
    return StratifiedGroupKFold(n_splits=max(2, min(3, n_splits)), shuffle=True, random_state=seed0)

# -------- Model factory (tanpa class_weight agar efek OS terlihat) --------
def make_model(name):
    if name == "LinearSVC":
        return LinearSVC(C=1.0, class_weight=None, max_iter=5000, random_state=RANDOM_STATE)
    if name == "Ridge":
        return RidgeClassifier(alpha=1.0, class_weight=None, random_state=RANDOM_STATE)
    raise ValueError("Unknown model name")

# -------- Oversampling di TRAIN (minority > majority) --------
def oversample_after_subset(X_tr_csr, y_tr, method: str, multiplier: float, use_tomek: bool, k_max=5):
    cnt = Counter(y_tr)
    if len(cnt) < 2:
        # no oversampling possible, just scale
        scaler = StandardScaler(with_mean=True, with_std=True)
        Xd = X_tr_csr.toarray() if hasattr(X_tr_csr, "toarray") else np.asarray(X_tr_csr)
        Xs = scaler.fit_transform(Xd)
        return Xs, y_tr, scaler, {"before": cnt, "after": cnt, "k_used": 0}

    maj = max(cnt, key=cnt.get); minc = min(cnt, key=cnt.get)
    n_maj, n_min = cnt[maj], cnt[minc]
    target_min = int(np.ceil(multiplier * n_maj))

    scaler = StandardScaler(with_mean=True, with_std=True)
    Xd = X_tr_csr.toarray() if hasattr(X_tr_csr, "toarray") else np.asarray(X_tr_csr)
    Xs = scaler.fit_transform(Xd)

    k = max(1, min(k_max, n_min - 1))
    can_algo = (n_min >= 2)

    if method.upper() == "SMOTE" and can_algo:
        sampler = SMOTE(sampling_strategy={minc: target_min}, k_neighbors=k, random_state=RANDOM_STATE)
        X_os, y_os = sampler.fit_resample(Xs, y_tr)
    elif method.upper() == "ADASYN" and can_algo:
        sampler = ADASYN(sampling_strategy={minc: target_min}, n_neighbors=k, random_state=RANDOM_STATE)
        X_os, y_os = sampler.fit_resample(Xs, y_tr)
    else:
        ros = RandomOverSampler(sampling_strategy={minc: target_min}, random_state=RANDOM_STATE)
        X_os, y_os = ros.fit_resample(Xs, y_tr)

    if use_tomek:
        tl = TomekLinks(n_jobs=-1)
        X_os, y_os = tl.fit_resample(X_os, y_os)

    return X_os, y_os, scaler, {"before": cnt, "after": Counter(y_os), "k_used": k}

# -------- Hardness ranking: pilih majority “paling sulit” --------
def hardness_rank_majority(X_csr, y, take_n):
    # Latih classifier ringan balanced utk dapat margin
    base = RidgeClassifier(alpha=1.0, class_weight="balanced", random_state=RANDOM_STATE)
    Xd = X_csr.astype(np.float32)
    base.fit(Xd, y)
    scores = base.decision_function(Xd)  # >0 => pred 1
    # majority = label 0; paling sulit = margin dekat 0 (|score| kecil) namun di sisi 0
    idx_major = np.where(y == 0)[0]
    margins = np.abs(scores[idx_major])
    # ambil indeks dengan margin terkecil
    order = np.argsort(margins)
    take_n = int(min(len(order), max(1, np.floor(take_n))))
    keep_idx_major = idx_major[order[:take_n]]
    return keep_idx_major

# -------- HHO (sederhana; 1D: keep_ratio) --------
@dataclass
class HHOConfig:
    pop_size: int = 12
    iters: int = 25
    r_min: float = 0.2
    r_max: float = 0.9

def run_hho_optimize_keep_ratio(X_tr, y_tr, groups_tr, over_method, multiplier, model_name,
                                inner_splits=3, cfg=HHOConfig()):
    rng = np.random.RandomState(RANDOM_STATE)
    # inisialisasi populasi
    pop = rng.uniform(cfg.r_min, cfg.r_max, size=cfg.pop_size)
    fitness = np.full(cfg.pop_size, -np.inf, dtype=float)

    def fitness_of_ratio(ratio: float) -> Tuple[float, Dict]:
        # hitung berapa majority yang disimpan
        n_maj = int(np.sum(y_tr == 0))
        keep_n = int(np.ceil(ratio * n_maj))
        # pilih majority paling sulit
        keep_idx_major = hardness_rank_majority(X_tr, y_tr, take_n=keep_n)
        # bentuk subset train
        keep_mask = np.zeros_like(y_tr, dtype=bool)
        keep_mask[keep_idx_major] = True
        keep_mask |= (y_tr == 1)  # minority semuanya dipakai
        X_sub = X_tr[keep_mask]
        y_sub = y_tr[keep_mask]
        g_sub = groups_tr[keep_mask]

        # inner-CV untuk evaluasi fitness (GM utama; tie F1→BAS)
        cv_inner = make_group_cv(y_sub, g_sub, n_splits=min(inner_splits, 5), seed0=RANDOM_STATE)
        f1s, bass, gms = [], [], []
        for tr, va in cv_inner.split(np.zeros_like(y_sub), y_sub, g_sub):
            X_i_tr, X_i_va = X_sub[tr], X_sub[va]
            y_i_tr, y_i_va = y_sub[tr], y_sub[va]

            # oversample setelah subset (minority > majority)
            X_os, y_os, scaler, _ = oversample_after_subset(
                X_i_tr, y_i_tr, method=over_method, multiplier=multiplier, use_tomek=False
            )
            X_va_scaled = scaler.transform(X_i_va.toarray() if hasattr(X_i_va, "toarray") else np.asarray(X_i_va))

            clf = make_model(model_name)
            clf.fit(X_os, y_os)
            y_hat = clf.predict(X_va_scaled)

            f1, bas, gm = score_triple(y_i_va, y_hat)
            f1s.append(f1); bass.append(bas); gms.append(gm)

        # fitness: GM mean; tie-breakers
        return (float(np.mean(gms)), {
            "gm": float(np.mean(gms)), "f1": float(np.mean(f1s)), "bas": float(np.mean(bass)),
            "keep_ratio": float(ratio), "keep_n": int(keep_n)
        })

    # evaluasi awal
    best_val, best_meta = -np.inf, None
    for i in range(cfg.pop_size):
        val, meta = fitness_of_ratio(pop[i])
        fitness[i] = val
        if val > best_val:
            best_val, best_meta = val, meta

    # iterasi HHO (sederhana 1D)
    for t in range(1, cfg.iters+1):
        E = 2 * (1 - t / cfg.iters)  # energy factor
        for i in range(cfg.pop_size):
            r = pop[i]
            q = rng.rand()
            if abs(E) >= 1:  # exploration
                r_new = best_meta["keep_ratio"] + rng.uniform(-1,1) * abs(best_meta["keep_ratio"] - r)
            else:            # exploitation
                if q >= 0.5:
                    r_new = best_meta["keep_ratio"] - E * abs(best_meta["keep_ratio"] - r)
                else:
                    r_new = best_meta["keep_ratio"] + E * abs(best_meta["keep_ratio"] - r)
            # clamp
            r_new = float(np.clip(r_new, cfg.r_min, cfg.r_max))
            # evaluate
            val, meta = fitness_of_ratio(r_new)
            # greedy accept
            if val > fitness[i]:
                pop[i] = r_new
                fitness[i] = val
                if val > best_val:
                    best_val, best_meta = val, meta

    return best_meta  # {"gm","f1","bas","keep_ratio","keep_n"}

# -------- Main outer loop: per label --------
rows = []
N_SPLITS = 5
for lab in SELECTED_LABELS:
    y_all = df[lab].astype(int).values
    cv_outer = make_group_cv(y_all, GROUPS, n_splits=N_SPLITS, seed0=RANDOM_STATE)

    # ambil winner config utk label ini
    win = WINNERS.get(lab, {"over":"ADASYN","model":"LinearSVC","multiplier":1.30,"use_tomek":False})
    over_m, model_m = win["over"], win["model"]
    mult_m, use_tomek = float(win["multiplier"]), bool(win.get("use_tomek", False))

    f1s, bass, gms = [], [], []
    folds_meta = []
    for fold_id, (tr, va) in enumerate(cv_outer.split(np.zeros_like(y_all), y_all, GROUPS), start=1):
        X_tr, X_va = X_INT[tr], X_INT[va]
        y_tr, y_va = y_all[tr], y_all[va]
        g_tr = GROUPS[tr]

        # 1) HHO: cari keep_ratio terbaik di TRAIN (inner-CV)
        best_meta = run_hho_optimize_keep_ratio(
            X_tr, y_tr, g_tr, over_method=over_m, multiplier=mult_m, model_name=model_m,
            inner_splits=3, cfg=HHOConfig(pop_size=12, iters=25, r_min=0.2, r_max=0.9)
        )

        # 2) Bangun TRAIN final dengan keep_ratio terbaik
        keep_n = best_meta["keep_n"]
        keep_idx_major = hardness_rank_majority(X_tr, y_tr, take_n=keep_n)
        keep_mask = np.zeros_like(y_tr, dtype=bool)
        keep_mask[keep_idx_major] = True
        keep_mask |= (y_tr == 1)
        X_sub, y_sub = X_tr[keep_mask], y_tr[keep_mask]

        # 3) Oversampling (minority > majority), optional Tomek
        X_os, y_os, scaler, info = oversample_after_subset(
            X_sub, y_sub, method=over_m, multiplier=mult_m, use_tomek=use_tomek
        )
        X_va_scaled = scaler.transform(X_va.toarray() if hasattr(X_va, "toarray") else np.asarray(X_va))

        # 4) Train final & eval di VALID outer
        clf = make_model(model_m)
        clf.fit(X_os, y_os)
        y_hat = clf.predict(X_va_scaled)

        f1, bas, gm = score_triple(y_va, y_hat)
        f1s.append(f1); bass.append(bas); gms.append(gm)

        folds_meta.append({
            "Fold": fold_id,
            "Best_keep_ratio": best_meta["keep_ratio"],
            "Best_inner_GM": best_meta["gm"],
            "Train_before_pos": int(np.sum(y_tr==1)),
            "Train_before_neg": int(np.sum(y_tr==0)),
            "Train_after_keep_neg": int(keep_n),
            "OS_after_pos": int(info["after"].get(1,0)),
            "OS_after_neg": int(info["after"].get(0,0)),
        })

        print(f"[HHO][{lab}] Fold{fold_id}: keep_ratio={best_meta['keep_ratio']:.3f} | "
              f"innerGM={best_meta['gm']:.4f} | "
              f"F1={f1:.4f} BAS={bas:.4f} GM={gm:.4f} | "
              f"before(+/−)={np.sum(y_tr==1)}/{np.sum(y_tr==0)} → keep_neg={keep_n} → "
              f"OS(+/−)={info['after'].get(1,0)}/{info['after'].get(0,0)}")

    # ringkasan outer
    rows.append({
        "Method_Over": over_m,
        "Label": lab,
        "Under": "HHO",
        "Valid_GM": float(np.mean(gms)),
        "Valid_BAS": float(np.mean(bass)),
        "Valid_F1": float(np.mean(f1s)),
        "Groups": int(len(np.unique(GROUPS))),
        "BestFitness(innerGM)_mean": float(np.mean([m["Best_inner_GM"] for m in folds_meta])),
        "Best_keep_ratio_mean": float(np.mean([m["Best_keep_ratio"] for m in folds_meta])),
        "MinorityMultiplier": mult_m,
        "APPLY_TOMEK": use_tomek,
        "Model": model_m
    })

res_hho = pd.DataFrame(rows)

# -------- Simpan hasil --------
from datetime import datetime
_ts = globals().get("_ts", datetime.now().strftime("%Y%m%d-%H%M%S"))
out_csv = f"hho_refinement_results_{_ts}.csv"
res_hho.to_csv(out_csv, index=False)
print(f"[STEP9] Hasil HHO refinement disimpan: {out_csv}")

# -------- Tampilkan ringkasan --------
def fmt(x): return f"{x:.4f}"
view = res_hho.copy()
view["Valid_GM"]  = view["Valid_GM"].map(fmt)
view["Valid_BAS"] = view["Valid_BAS"].map(fmt)
view["Valid_F1"]  = view["Valid_F1"].map(fmt)
view["BestFitness(innerGM)_mean"] = view["BestFitness(innerGM)_mean"].map(fmt)
view = view[["Method_Over","Label","Under","Model","MinorityMultiplier",
             "Best_keep_ratio_mean","BestFitness(innerGM)_mean",
             "Valid_GM","Valid_BAS","Valid_F1","APPLY_TOMEK"]]
try:
    from IPython.display import display
    display(view)
except Exception:
    print(view.to_string(index=False))

[HHO][product issues] Fold1: keep_ratio=0.605 | innerGM=0.1637 | F1=0.0000 BAS=0.4947 GM=0.0000 | before(+/−)=17/1124 → keep_neg=681 → OS(+/−)=886/681
[HHO][product issues] Fold2: keep_ratio=0.564 | innerGM=0.1659 | F1=0.0000 BAS=0.4912 GM=0.0000 | before(+/−)=19/1122 → keep_neg=633 → OS(+/−)=823/633
[HHO][product issues] Fold3: keep_ratio=0.462 | innerGM=0.0000 | F1=0.0000 BAS=0.4946 GM=0.0000 | before(+/−)=16/1126 → keep_neg=521 → OS(+/−)=678/521
[HHO][product issues] Fold4: keep_ratio=0.386 | innerGM=0.1149 | F1=0.0000 BAS=0.4875 GM=0.0000 | before(+/−)=18/1124 → keep_neg=435 → OS(+/−)=566/435
[HHO][product issues] Fold5: keep_ratio=0.462 | innerGM=0.0000 | F1=0.0000 BAS=0.4982 GM=0.0000 | before(+/−)=18/1124 → keep_neg=520 → OS(+/−)=676/520


/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  

[HHO][neoplasms benign, malignant and unspecified (incl cysts and polyps)] Fold1: keep_ratio=0.876 | innerGM=0.6116 | F1=0.4331 BAS=0.6155 GM=0.5970 | before(+/−)=303/838 → keep_neg=735 → OS(+/−)=956/735


/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  

[HHO][neoplasms benign, malignant and unspecified (incl cysts and polyps)] Fold2: keep_ratio=0.889 | innerGM=0.6059 | F1=0.4403 BAS=0.6180 GM=0.6007 | before(+/−)=302/840 → keep_neg=747 → OS(+/−)=972/747


/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/gibannn/miniconda3/envs/BIOINFORMATIKS/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  